# 실습 1주차: 데이터와 모형 — 좋은 파라미터를 직접 찾아낸다

> **시나리오 — 오늘 만들 것**
>
>
> Ch01의 인터랙티브에서 슬라이더로 직선을 맞춰 보았다. **오늘 그것을 코드로 한다.**
>
> 펭귄 333마리의 **날개 길이**로 **몸무게**를 예측하는 모형을 만들고,
> 잔차 제곱합이 가장 작아지는 **파라미터를 직접 찾아낸다.**
>
> $$\text{데이터} \;\to\; \text{모형}(w, b) \;\to\; \text{예측} \;\to\; \text{얼마나 틀렸나} \;\to\; \text{더 나은 } w, b$$
>
> - **대응 이론**: [Ch01 들어가기: 데이터와 모형](ch01.qmd)
> - 코드는 완성되어 있다. **직접 해보기** 칸은 스스로 채운 뒤 아래 정답과 맞춰 본다. 채점하지 않는다.


> **오늘 배우는 PyTorch 부품**
>
>
> | 부품 | 이론에서의 이름 |
> |------|------|
> | `torch.tensor` | 데이터를 담는 그릇 |
> | `TensorDataset` | 데이터셋 — `len`, 한 건 꺼내기 |
> | `nn.Module` · `nn.Parameter` | **모형**과 **파라미터** |
> | `forward()` | 모형이 예측을 만드는 식 |
> | `nn.MSELoss` | 얼마나 틀렸는지 재는 자 |
>
> 이 부품들은 **버리지 않고 15주 내내 그대로 쓴다.** 오늘 만드는 모형이 앞으로 만들 모든 모형의 원형이다.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset

torch.manual_seed(42)
print('torch', torch.__version__)

---

# 1. 데이터 열기

## 1-1. 인터넷에서 바로 불러온다

In [ ]:
URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
df = pd.read_csv(URL)
df.head()

## 1-2. 무엇이 들어 있나

In [ ]:
print('행, 열 :', df.shape)
print()
print(df.dtypes)

숫자 열(`float64`)과 글자 열(`object`)이 섞여 있다.
Ch01에서 말한 **정형 데이터** — 행이 데이터 포인트, 열이 변수다.

## 1-3. 빈칸이 있는 행은 뺀다

In [ ]:
print('제거 전 :', len(df))
print('열별 결측 수:\n', df.isna().sum())

d = df.dropna().reset_index(drop=True)
print('\n제거 후 :', len(d))

## 1-4. 그림으로 먼저 본다

In [ ]:
plt.figure(figsize=(5.5, 3.8))
plt.scatter(d['flipper_length_mm'], d['body_mass_g'], s=14, alpha=0.6)
plt.xlabel('flipper length (mm)'); plt.ylabel('body mass (g)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

날개가 길수록 무겁다. **직선 하나로 꽤 설명될 것 같다** — 오늘 찾을 것이 그 직선이다.

> **직접 해보기 ① — 다른 변수로 그려 보기**
>
>
> `bill_length_mm` 와 `body_mass_g` 의 산점도를 그려 보시오.
> 날개 길이만큼 뚜렷한 관계인가?

In [ ]:
# ✏️ 직접 채워 보세요
plt.figure(figsize=(5.5, 3.8))
plt.scatter(...)          # ← 여기를 채우세요
plt.xlabel('bill length (mm)'); plt.ylabel('body mass (g)')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
plt.figure(figsize=(5.5, 3.8))
plt.scatter(d['bill_length_mm'], d['body_mass_g'], s=14, alpha=0.6, color='C1')
plt.xlabel('bill length (mm)'); plt.ylabel('body mass (g)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

관계가 있긴 하지만 **훨씬 흩어져 있다.** 변수마다 예측에 주는 도움이 다르다.

---

# 2. 파이토치가 읽는 형태로 — 텐서와 `Dataset`

## 2-1. `X` 와 `y` 를 텐서로

In [ ]:
X = torch.tensor(d[['flipper_length_mm']].to_numpy(dtype='float32'))
y = torch.tensor(d['body_mass_g'].to_numpy(dtype='float32')).unsqueeze(1)

print('X :', tuple(X.shape), X.dtype, '  ← (n, p) = (데이터 포인트 수, 변수 수)')
print('y :', tuple(y.shape), y.dtype, '  ← 세로로 세운다')
print()
print('X 앞 3개:\n', X[:3])
print('y 앞 3개:\n', y[:3])

> **두 가지를 항상 맞춘다**
>
>
> **① `dtype` 은 `float32`** — numpy의 기본은 `float64`인데 PyTorch 신경망은 `float32`를 쓴다.
> 안 맞으면 `expected Float but found Double` 에러가 난다.
>
> **② `y` 는 `(n, 1)` 로 세운다** — `.unsqueeze(1)` 이 `(n,)` 을 `(n, 1)` 로 바꾼다.
> 모형의 출력이 `(n, 1)` 이므로 정답도 같은 모양이어야 손실이 제대로 계산된다.


## 2-2. `TensorDataset` — 데이터를 파이토치 방식으로 담는다

In [ ]:
ds = TensorDataset(X, y)

print('데이터 수 len(ds) :', len(ds))
print('0번 데이터 ds[0]  :', ds[0])

x0, y0 = ds[0]
print('\n0번 펭귄의 날개 :', float(x0), 'mm')
print('0번 펭귄의 몸무게:', float(y0), 'g')

> `Dataset` 은 **"몇 건이 있고, i번째를 어떻게 꺼내는가"** 두 가지만 아는 객체다.
> 사진이든 문장이든 전부 이 형태로 감싸서 다룬다 — 5주차에 이미지를, 10주차에 문장을 같은 방식으로 담는다.


> **직접 해보기 ② — 변수 2개짜리 데이터셋**
>
>
> 부리 길이와 날개 길이 **두 개**를 입력으로 하는 텐서 `X2` 와 데이터셋 `ds2` 를 만드시오.
> `X2` 의 shape은 `(333, 2)` 여야 한다.

In [ ]:
# ✏️ 직접 채워 보세요
X2 = None        # ← 여기를 채우세요
ds2 = None       # ← TensorDataset(...)

assert X2 is not None and tuple(X2.shape) == (333, 2), f'shape이 다릅니다'
assert len(ds2) == 333
print('통과  X2', tuple(X2.shape), ' ds2[0]', ds2[0][0].tolist())

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
X2 = torch.tensor(d[['bill_length_mm', 'flipper_length_mm']].to_numpy(dtype='float32'))
ds2 = TensorDataset(X2, y)
print('X2', tuple(X2.shape), ' ds2[0]', ds2[0][0].tolist())

---

# 3. 모형 — `nn.Module`

Ch01의 정의 그대로다. **모형은 파라미터를 가진 함수**다.

$$\hat{y} = w x + b$$

PyTorch에서 모형은 `nn.Module` 을 상속받아 두 가지를 적는다.

- `__init__` : **어떤 파라미터를 가지는가**
- `forward` : **입력을 받아 어떻게 예측을 만드는가**

## 3-1. 직접 만들어 본다

In [ ]:
class LinearModel(nn.Module):
    def __init__(self, p):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(p, 1))   # 가중치 — 학습 대상
        self.b = nn.Parameter(torch.zeros(1))      # 편향 — 학습 대상

    def forward(self, x):
        return x @ self.w + self.b                 # 예측식 그대로

model = LinearModel(p=1)
print(model)

In [ ]:
for name, param in model.named_parameters():
    print(f'{name:4s} {str(tuple(param.shape)):8s} 값 {param.data.numpy().ravel()}')
print('\n파라미터 수:', sum(p.numel() for p in model.parameters()))

> **`nn.Parameter` 가 곧 Ch01의 **파라미터**다**
>
>
> `nn.Parameter` 로 선언한 텐서만 **모형의 파라미터**로 등록된다.
> `model.parameters()` 로 목록을 꺼낼 수 있고, 3주차부터는 이 목록이 그대로 학습 대상이 된다.
>
> 파라미터는 **데이터로부터 정해지는 값**이고, 사람이 정하는 값(변수를 몇 개 쓸지, 나중에 배울 학습률)은
> **하이퍼파라미터**다 — Ch01에서 구분한 그것이다.


## 3-2. 예측해 본다

In [ ]:
with torch.no_grad():
    model.w.fill_(50.0)      # 날개 1mm당 50g 이라고 가정
    model.b.fill_(-5800.0)

pred = model(X)
print('예측 shape :', tuple(pred.shape))
print()
print(pd.DataFrame({
    '날개(mm)': X[:5, 0].numpy(),
    '예측(g)': pred[:5, 0].detach().numpy().round(1),
    '실제(g)': y[:5, 0].numpy(),
}).to_string(index=False))

`model(X)` 한 번으로 **333마리 전부**가 계산된다. 반복문이 필요 없다.

In [ ]:
with torch.no_grad():
    loop = torch.stack([X[i] @ model.w + model.b for i in range(len(X))])
print('반복문 결과 shape:', tuple(loop.shape))
print('한 번에 계산한 것과 같은가:', torch.allclose(loop, pred, atol=1e-3))

## 3-3. 파라미터를 바꾸면 예측이 바뀐다

Ch01의 후보 A·B·C 표를 코드로 확인한다.

In [ ]:
grid_x = torch.linspace(X.min(), X.max(), 100).unsqueeze(1)

plt.figure(figsize=(5.8, 3.9))
plt.scatter(X, y, s=14, alpha=0.35, color='gray', label='data')
for w_, b_, name in [(30.0, -1500.0, 'A'), (50.0, -5800.0, 'B'), (70.0, -10500.0, 'C')]:
    with torch.no_grad():
        model.w.fill_(w_); model.b.fill_(b_)
        plt.plot(grid_x, model(grid_x), lw=2, label=f'{name}: w={w_:.0f}, b={b_:.0f}')
plt.xlabel('flipper length (mm)'); plt.ylabel('body mass (g)')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

세 직선 중 **어느 것이 좋은가?** 눈으로는 대충 알겠지만, **숫자로 말할 수 있어야** 컴퓨터에게 시킬 수 있다.

> **직접 해보기 ③ — 입력 2개짜리 모형**
>
>
> `LinearModel(p=2)` 를 만들어 `X2` 를 통과시키고 출력 shape과 파라미터 수를 확인하시오.

In [ ]:
# ✏️ 직접 채워 보세요
model2 = None            # ← LinearModel(...)
out2 = None              # ← model2(X2)

assert out2 is not None and tuple(out2.shape) == (333, 1)
print('출력', tuple(out2.shape),
      ' 파라미터', sum(p.numel() for p in model2.parameters()))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
model2 = LinearModel(p=2)
out2 = model2(X2)
print('출력', tuple(out2.shape),
      ' 파라미터', sum(p.numel() for p in model2.parameters()), '= 가중치 2 + 편향 1')

---

# 4. 좋은 파라미터란 — 잔차 제곱합

Ch01의 인터랙티브가 말한 그 값이다.

$$L = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

## 4-1. `nn.MSELoss`

In [ ]:
criterion = nn.MSELoss()

with torch.no_grad():
    for w_, b_, name in [(30.0, -1500.0, 'A'), (50.0, -5800.0, 'B'), (70.0, -10500.0, 'C')]:
        model.w.fill_(w_); model.b.fill_(b_)
        mse = criterion(model(X), y)
        print(f'{name}: w={w_:5.0f}  b={b_:8.0f}   MSE {float(mse):12,.0f}   RMSE {float(mse.sqrt()):7.1f} g')

**B가 가장 좋다.** 이제 "좋다"를 숫자로 말할 수 있다.

## 4-2. 직접 계산과 대조

In [ ]:
with torch.no_grad():
    model.w.fill_(50.0); model.b.fill_(-5800.0)
    pred = model(X)

manual = ((y - pred) ** 2).mean()
print('직접 계산  :', float(manual))
print('nn.MSELoss :', float(criterion(pred, y)))
print()
print('RMSE :', round(float(manual.sqrt()), 1), 'g  ← 평균적으로 이만큼 틀린다')
print('몸무게 표준편차:', round(float(y.std(unbiased=False)), 1), 'g  ← 아무것도 안 하면 이만큼 틀린다')

RMSE는 MSE의 제곱근이다. **단위가 원래대로(g) 돌아와** 해석하기 쉽다.

> **직접 해보기 ④ — RMSE 함수를 직접 만들기**
>
>
> 모형과 데이터를 받아 **RMSE** 를 돌려주는 함수를 작성하시오.

In [ ]:
# ✏️ 직접 채워 보세요
def rmse(model, X, y):
    return None            # ← 여기를 채우세요

with torch.no_grad():
    model.w.fill_(50.0); model.b.fill_(-5800.0)
    got = rmse(model, X, y)

assert got is not None, '아직 채우지 않았습니다'
assert abs(float(got) - 394.3) < 1.0, f'값이 이상합니다: {float(got)}'
print('통과  RMSE =', round(float(got), 1))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
def rmse(model, X, y):
    with torch.no_grad():
        return criterion(model(X), y).sqrt()

model.w.data.fill_(50.0); model.b.data.fill_(-5800.0)
print('RMSE =', round(float(rmse(model, X, y)), 1))

---

# 5. 완성 — 최적 파라미터를 직접 찾아낸다

Ch01의 학습 정의: **"데이터를 가장 잘 나타내는 파라미터를 찾아나가는 과정."**
가장 단순한 방법은 **후보를 전부 넣어 보는 것**이다.

## 5-1. 파라미터 하나를 훑는다

편향 `b` 를 −5800으로 고정하고 기울기 `w` 만 0~120까지 훑는다.

In [ ]:
import time

ws = torch.linspace(0, 120, 121)
losses = []
t0 = time.time()
with torch.no_grad():
    model.b.fill_(-5800.0)
    for w_ in ws:
        model.w.fill_(float(w_))
        losses.append(float(criterion(model(X), y)))
elapsed_1d = time.time() - t0

best_i = int(np.argmin(losses))
print(f'{len(ws)}가지를 시도  {elapsed_1d:.3f}초')
print(f'가장 좋은 w = {float(ws[best_i]):.0f}   RMSE {losses[best_i]**0.5:.1f} g')

In [ ]:
plt.figure(figsize=(5.8, 3.5))
plt.plot(ws, np.sqrt(losses))
plt.axvline(float(ws[best_i]), color='red', ls='--', label=f'best w = {float(ws[best_i]):.0f}')
plt.xlabel('w (weight of flipper length)'); plt.ylabel('RMSE (g)')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

**곡선의 바닥을 찾는 것** — 이것이 학습이다.

## 5-2. 두 개를 격자로 훑는다

이번에는 `w` 와 `b` 를 **동시에** 움직인다.

In [ ]:
ws = torch.linspace(0, 120, 121)
bs = torch.linspace(-8000, 2000, 101)
L = np.zeros((len(bs), len(ws)))

t0 = time.time()
with torch.no_grad():
    for i, b_ in enumerate(bs):
        for j, w_ in enumerate(ws):
            model.w.fill_(float(w_)); model.b.fill_(float(b_))
            L[i, j] = float(criterion(model(X), y))
elapsed_2d = time.time() - t0

bi, bj = np.unravel_index(np.argmin(L), L.shape)
best_w, best_b = float(ws[bj]), float(bs[bi])
print(f'{len(ws)*len(bs):,}가지를 시도  {elapsed_2d:.1f}초')
print(f'가장 좋은 w = {best_w:.0f},  b = {best_b:.0f}   RMSE {L[bi, bj]**0.5:.1f} g')

In [ ]:
plt.figure(figsize=(6, 4))
cs = plt.contourf(ws, bs, np.sqrt(L), levels=30, cmap='viridis')
plt.colorbar(cs, label='RMSE (g)')
plt.plot(best_w, best_b, 'r*', ms=16, label=f'최적 ({best_w:.0f}, {best_b:.0f})')
plt.xlabel('w'); plt.ylabel('b'); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

손실이 **그릇 모양**이고, 바닥에 최적점 하나가 있다.
Ch01의 슬라이더가 하던 일을 12,221번 자동으로 한 것이다.

## 5-3. 찾은 모형으로 예측한다

In [ ]:
with torch.no_grad():
    model.w.fill_(best_w); model.b.fill_(best_b)
    pred = model(X)

print(f'모형 : 몸무게 = {best_w:.0f} x 날개길이 + ({best_b:.0f})')
print(f'RMSE : {float(rmse(model, X, y)):.1f} g')
print()
print(f'해석 : 날개가 1mm 길수록 몸무게가 약 {best_w:.0f}g 무겁다')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.9))
axes[0].scatter(X, y, s=14, alpha=0.4, color='gray')
axes[0].plot(grid_x, model(grid_x).detach(), 'r-', lw=2)
axes[0].set_xlabel('flipper length (mm)'); axes[0].set_ylabel('body mass (g)')
axes[0].grid(alpha=0.3)

axes[1].scatter(y, pred.detach(), s=14, alpha=0.6)
lim = [float(y.min()) - 200, float(y.max()) + 200]
axes[1].plot(lim, lim, 'r--', lw=1.5)
axes[1].set_xlim(lim); axes[1].set_ylim(lim)
axes[1].set_xlabel('actual (g)'); axes[1].set_ylabel('predicted (g)')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

> **직접 해보기 ⑤ — 다른 변수로 같은 일을 하기**
>
>
> 날개 길이 대신 **부리 길이**로 같은 격자 탐색을 하여 최적 `w`, `b` 와 RMSE를 구하시오.
> 1-4절에서 본 대로, 어느 쪽이 더 잘 맞히는가?

In [ ]:
# ✏️ 직접 채워 보세요
Xb = None            # ← bill_length_mm 로 만든 (333, 1) 텐서
mb = LinearModel(p=1)

best = (float('inf'), 0, 0)
with torch.no_grad():
    for b_ in torch.linspace(0, 6000, 101):
        for w_ in torch.linspace(0, 200, 121):
            ...      # ← 파라미터를 채우고 손실을 재서 best 를 갱신하세요
print(best)

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
Xb = torch.tensor(d[['bill_length_mm']].to_numpy(dtype='float32'))
mb = LinearModel(p=1)

best = (float('inf'), 0.0, 0.0)
with torch.no_grad():
    for b_ in torch.linspace(0, 6000, 101):
        for w_ in torch.linspace(0, 200, 121):
            mb.w.fill_(float(w_)); mb.b.fill_(float(b_))
            v = float(criterion(mb(Xb), y))
            if v < best[0]:
                best = (v, float(w_), float(b_))
print(f'부리 길이 : w={best[1]:.0f}  b={best[2]:.0f}  RMSE {best[0]**0.5:.1f} g')
print(f'날개 길이 : w={best_w:.0f}  b={best_b:.0f}  RMSE {L[bi, bj]**0.5:.1f} g')

산점도에서 본 그대로 **날개 길이가 훨씬 잘 맞힌다.**
어떤 변수를 쓸 것인가도 모형 설계의 일부다.

---

# 6. 이 방법의 한계 — 왜 3주차가 필요한가

## 6-1. 파라미터가 하나 늘 때마다

In [ ]:
per_eval = elapsed_2d / (len(ws) * len(bs))

rows = []
for n_param, label in [(1, '기울기만'), (2, '기울기 + 편향'), (3, '변수 2개 모형'),
                       (4, '변수 3개 모형'), (10, '작은 신경망')]:
    trials = 100 ** n_param
    sec = trials * per_eval
    if sec < 60:      t = f'{sec:.1f}초'
    elif sec < 3600:  t = f'{sec/60:.1f}분'
    elif sec < 86400: t = f'{sec/3600:.1f}시간'
    else:             t = f'{sec/86400/365:,.0f}년'
    rows.append({'파라미터 수': n_param, '모형': label,
                 '격자 후보 (칸당 100)': f'{trials:.0e}', '예상 시간': t})
print(pd.DataFrame(rows).to_string(index=False))

> **훑어서는 안 된다**
>
>
> 파라미터가 **하나 늘 때마다 후보가 100배**가 된다.
> 12주차에 만들 GPT-2 small은 파라미터가 **1억 2천만 개**다.
>
> $$100^{124{,}439{,}808}$$
>
> 우주의 원자 수보다 많다. **다른 방법이 필요하다.**


## 6-2. 다음 주부터 할 일

In [ ]:
plt.figure(figsize=(5.8, 3.5))
plt.plot(ws, np.sqrt(L[bi, :]))
i0 = 20
plt.plot(ws[i0], np.sqrt(L[bi, i0]), 'ro', ms=9)
plt.annotate('여기 서 있다면\n어느 쪽으로 가야 하나?',
             xy=(float(ws[i0]), np.sqrt(L[bi, i0])), xytext=(60, 900),
             arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')
plt.xlabel('w'); plt.ylabel('RMSE (g)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

전부 훑지 않고 **지금 서 있는 자리에서 내려가는 방향만** 알면 된다.
그 방향을 알려 주는 것이 **기울기(미분)** 이고, 그것으로 파라미터를 고쳐 나가는 것이
**경사하강법** — **3주차**에 배운다.

> 그리고 하나 더. 오늘 우리는 **333마리 전부로 파라미터를 찾고, 같은 333마리로 점수를 매겼다.**
> 이것이 왜 위험한지는 **4주차**에서 다룬다.


---

# 7. 정리

> **이번 주 체크포인트**
>
>
> | 하고 싶은 일 | 코드 |
> |------|------|
> | 데이터 불러오기 | `pd.read_csv(URL)` → `df.dropna()` |
> | 텐서로 | `torch.tensor(arr.astype('float32'))` |
> | 정답 세우기 | `y.unsqueeze(1)` → `(n, 1)` |
> | 데이터셋 | `TensorDataset(X, y)` — `len(ds)`, `ds[i]` |
> | 모형 정의 | `class M(nn.Module)` + `__init__` + `forward` |
> | 파라미터 선언 | `nn.Parameter(torch.zeros(...))` |
> | 파라미터 목록 | `model.named_parameters()`, `model.parameters()` |
> | 파라미터 수 | `sum(p.numel() for p in model.parameters())` |
> | 예측 | `model(X)` — 전부 한 번에 |
> | 손실 | `nn.MSELoss()(pred, y)` |
> | 값 직접 바꾸기 | `with torch.no_grad(): model.w.fill_(...)` |


**오늘의 결론**

$$\text{모형} = \text{파라미터를 가진 함수}, \qquad \text{학습} = \text{손실을 가장 작게 하는 파라미터 찾기}$$

찾는 방법이 오늘은 **전부 훑기**였고, 3주차부터는 **내려가는 방향으로 걷기**가 된다.

## 스스로 확인해 보기

아래 결과를 먼저 예상한 뒤 실행해서 대조한다.

In [ ]:
m = LinearModel(p=3)
xb = torch.randn(7, 3)

print('파라미터 목록:')
for n_, p_ in m.named_parameters():
    print(f'  {n_:4s} {tuple(p_.shape)}')
print('파라미터 수  :', sum(p.numel() for p in m.parameters()))
print('출력 shape   :', tuple(m(xb).shape))
print()
yb = torch.randn(7, 1)
ds3 = TensorDataset(xb, yb)
print('len(ds3)     :', len(ds3))
print('ds3[2][0]    :', ds3[2][0].numpy().round(3))
print()
with torch.no_grad():
    m.w.fill_(1.0); m.b.fill_(0.0)
print('w=1, b=0 일 때 MSELoss :', round(float(nn.MSELoss()(m(xb), yb)), 4))
with torch.no_grad():
    m.w.fill_(0.0)
print('w=0, b=0 일 때 MSELoss :', round(float(nn.MSELoss()(m(xb), yb)), 4))

---

## 다음 실습

[실습 2주차: 퍼셉트론을 쌓아 신경망 만들기](lab02.qmd) —
오늘 직접 만든 `LinearModel` 이 사실 PyTorch에 이미 있다(`nn.Linear`).
그것을 **여러 개, 여러 층**으로 쌓으면 직선으로 안 되던 문제가 풀리기 시작한다.